# Food Delivery Time Prediction using CNN and Validation Techniques

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.linear_model import LogisticRegression

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

df = pd.read_csv('Food_Delivery_Time_Prediction 3.csv')
print(df.shape)
df.head()


## Dataset Columns

In [ ]:
print(['Order_ID', 'Customer_Location', 'Restaurant_Location', 'Distance', 'Weather_Conditions', 'Traffic_Conditions', 'Delivery_Person_Experience', 'Order_Priority', 'Order_Time', 'Vehicle_Type', 'Restaurant_Rating', 'Customer_Rating', 'Delivery_Time', 'Order_Cost', 'Tip_Amount'])

## Data Cleaning and Preprocessing

In [ ]:

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(exclude='object').columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()


## Feature Engineering

In [ ]:

if 'Order_Time' in df.columns:
    dt = pd.to_datetime(df['Order_Time'], errors='coerce')
    df['Rush_Hour'] = dt.dt.hour.isin([8,9,13,14,18,19,20]).astype(int)

if 'Delivery_Time' in df.columns:
    threshold = df['Delivery_Time'].median()
    df['Delivery_Status'] = (df['Delivery_Time'] > threshold).astype(int)

df.head()


## Scaling

In [ ]:

target = 'Delivery_Status'

X = df.drop(columns=[target])
y = df[target]

scaler = StandardScaler()
X = scaler.fit_transform(X)


## CNN Model (1D CNN for Tabular Features)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

model = Sequential([
    Conv1D(32, 3, activation='relu', input_shape=(X_train.shape[1],1)),
    MaxPooling1D(2),
    Conv1D(64, 3, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(
    X_train_cnn, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=16,
    verbose=1
)

loss, acc = model.evaluate(X_test_cnn, y_test)
print("CNN Accuracy:", acc)


In [ ]:

pred_prob = model.predict(X_test_cnn)
pred = (pred_prob > 0.5).astype(int)

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

fpr, tpr, _ = roc_curve(y_test, pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr)
plt.title(f'ROC Curve (AUC={roc_auc:.3f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()


## Logistic Regression Baseline

In [ ]:

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print(classification_report(y_test, pred_lr))


## K-Fold Cross Validation

In [ ]:

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X,
    y,
    cv=kf,
    scoring='accuracy'
)

print("Fold Accuracies:", scores)
print("Mean Accuracy:", scores.mean())



# Final Report

## Methodology
- Data preprocessing and encoding
- Feature engineering (rush hour and delivery status)
- 1D CNN classification model
- Logistic Regression comparison
- K-Fold validation
- ROC and confusion matrix evaluation

## Metrics
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

## Recommendations
- Use traffic, weather, and rush-hour features to reduce delays.
- Deploy predictive models for delivery assignment optimization.
- Compare CNN performance against traditional machine learning models.
